In [ ]:
# ── Cell 1: Imports & Configuration ──────────────────────────────────────────
import requests
import pandas as pd
from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter
from openpyxl.worksheet.table import Table, TableStyleInfo
from datetime import datetime
import time
import warnings
warnings.filterwarnings('ignore')

# ── Only these two variables need to change ───────────────────────────────────
TICKER     = "AAPL"
USER_AGENT = "Your Name your@email.com"   # Required by SEC fair-use policy
# ─────────────────────────────────────────────────────────────────────────────

TODAY       = datetime.today().strftime('%Y-%m-%d')
OUTPUT_FILE = f"{TICKER}_10K_Historical_{TODAY}.xlsx"

In [ ]:
# ── Cell 2: Helper Functions ──────────────────────────────────────────────────

HEADERS     = {"User-Agent": USER_AGENT, "Accept-Encoding": "gzip, deflate", "Host": "data.sec.gov"}
HEADERS_SEC = {"User-Agent": USER_AGENT, "Accept-Encoding": "gzip, deflate", "Host": "www.sec.gov"}

def fetch_json(url, retries=3):
    h = HEADERS if "data.sec.gov" in url else HEADERS_SEC
    for attempt in range(retries):
        try:
            time.sleep(0.15)
            r = requests.get(url, headers=h, timeout=30)
            r.raise_for_status()
            return r.json()
        except Exception as e:
            if attempt < retries - 1:
                time.sleep(2 ** attempt)
            else:
                raise e

def to_millions(value, unit):
    if value is None:
        return None
    return round(value / 1_000_000, 2) if unit in ("USD", "shares") else value

def resolve_tag(facts_gaap, tag_candidates, fy_end_set, negate=False):
    """Try tags in ranked order; return ({fy_end: value}, tag_used) for the first hit."""
    for tag in tag_candidates:
        if tag not in facts_gaap:
            continue
        units_dict = facts_gaap[tag].get("units", {})
        unit_key   = next((k for k in units_dict if k in ("USD", "shares", "pure")), None)
        if unit_key is None:
            continue
        annual = [e for e in units_dict[unit_key] if e.get("form") in ("10-K", "10-K/A")]
        fy_map = {}
        for e in annual:
            end = e.get("end")
            if end not in fy_end_set:
                continue
            filed = e.get("filed", "")
            if end not in fy_map or filed > fy_map[end]["filed"]:
                fy_map[end] = {"val": e.get("val"), "unit": unit_key, "filed": filed}
        if not fy_map:
            continue
        result = {}
        for end, info in fy_map.items():
            v = to_millions(info["val"], info["unit"])
            result[end] = (-v if negate and v is not None else v)
        return result, tag
    return {}, None

def build_row(facts_gaap, tag_candidates, fy_ends, negate=False):
    data, tag_used = resolve_tag(facts_gaap, tag_candidates, set(fy_ends), negate=negate)
    return [data.get(fy) for fy in fy_ends], tag_used

In [ ]:
# ── Cell 3: CIK Lookup & 10-K Filing Identification ──────────────────────────

tickers_data = fetch_json("https://www.sec.gov/files/company_tickers.json")
ticker_upper = TICKER.upper()
cik_raw = company_name = None

for entry in tickers_data.values():
    if entry["ticker"].upper() == ticker_upper:
        cik_raw      = entry["cik_str"]
        company_name = entry["title"]
        break

if cik_raw is None:
    raise ValueError(f"Ticker '{TICKER}' not found in SEC ticker list")

CIK = str(cik_raw).zfill(10)
print(f"\u2705 Resolved {TICKER} \u2192 {company_name} (CIK: {CIK})")

submissions = fetch_json(f"https://data.sec.gov/submissions/CIK{CIK}.json")
filings     = submissions.get("filings", {}).get("recent", {})

tenk_filings = [
    {"accessionNumber": filings["accessionNumber"][i],
     "filingDate":      filings["filingDate"][i],
     "reportDate":      filings["reportDate"][i],
     "primaryDocument": filings["primaryDocument"][i]}
    for i, form in enumerate(filings.get("form", []))
    if form in ("10-K", "10-K/A")
]

tenk_filings.sort(key=lambda x: x["filingDate"], reverse=True)
tenk_filings = tenk_filings[:5]
tenk_filings.sort(key=lambda x: x["reportDate"])    # oldest → newest

FY_ENDS   = [f["reportDate"] for f in tenk_filings]
FY_LABELS = [f"FY{f['reportDate'][:4]}" for f in tenk_filings]

# Forecast labels start one year after the most recent 10-K
MOST_RECENT_YEAR = int(FY_ENDS[-1][:4])
PROJ_LABELS      = [f"FY{MOST_RECENT_YEAR + i}E" for i in range(1, 4)]

print(f"\u2705 Found {len(tenk_filings)} 10-K filings : {', '.join(FY_LABELS)}")
print(f"\u2705 Forecast columns              : {', '.join(PROJ_LABELS)}")

In [ ]:
# ── Cell 4: XBRL Company Facts Download ──────────────────────────────────────

facts_data = fetch_json(f"https://data.sec.gov/api/xbrl/companyfacts/CIK{CIK}.json")
facts_gaap = facts_data.get("facts", {}).get("us-gaap", {})
print(f"\u2705 Pulled XBRL company facts (us-gaap concepts: {len(facts_gaap)})")

In [ ]:
# ── Cell 5: Income Statement DataFrame ───────────────────────────────────────

IS_SPEC = [
    ("Revenue",               ["RevenueFromContractWithCustomerExcludingAssessedTax",
                               "Revenues", "SalesRevenueNet",
                               "RevenueFromContractWithCustomerIncludingAssessedTax"],   False, False),
    ("Cost of Revenue",       ["CostOfGoodsSold", "CostOfRevenue",
                               "CostOfGoodsAndServicesSold"],                             False, False),
    ("Gross Profit",          ["GrossProfit"],                                            False, "Revenue - Cost of Revenue"),
    ("R&D Expense",           ["ResearchAndDevelopmentExpense",
                               "ResearchAndDevelopmentExpenseExcludingAcquiredInProcessCost"], False, False),
    ("SG&A Expense",          ["SellingGeneralAndAdministrativeExpense",
                               "GeneralAndAdministrativeExpense"],                        False, False),
    ("Operating Income",      ["OperatingIncomeLoss",
                               "IncomeLossFromContinuingOperationsBeforeIncomeTaxes"],   False, False),
    ("Interest Expense",      ["InterestExpense", "InterestAndDebtExpense"],              False, False),
    ("Pretax Income",         ["IncomeLossFromContinuingOperationsBeforeIncomeTaxesExtraordinaryItemsNoncontrollingInterest",
                               "IncomeLossFromContinuingOperationsBeforeIncomeTaxesDomestic"], False, False),
    ("Income Tax Expense",    ["IncomeTaxExpenseBenefit", "CurrentIncomeTaxExpenseBenefit"], False, False),
    ("Net Income",            ["NetIncomeLoss",
                               "NetIncomeLossAvailableToCommonStockholdersBasic"],       False, False),
    ("EPS Basic (per share)", ["EarningsPerShareBasic"],                                  False, False),
    ("EPS Diluted (per share)",["EarningsPerShareDiluted"],                               False, False),
    ("Diluted Shares",        ["WeightedAverageNumberOfDilutedSharesOutstanding",
                               "CommonStockSharesOutstanding"],                          False, False),
    ("D&A",                   ["DepreciationDepletionAndAmortization",
                               "Depreciation", "DepreciationAndAmortization"],           False, False),
]

is_rows, is_tags, missing_is = {}, {}, []

for label, tags, negate, derived in IS_SPEC:
    if derived:
        vals, tag_used = build_row(facts_gaap, tags, FY_ENDS, negate)
        if tag_used is None:
            a_lbl, b_lbl = derived.split(" - ")
            a = is_rows.get(a_lbl, [None]*len(FY_ENDS))
            b = is_rows.get(b_lbl, [None]*len(FY_ENDS))
            vals     = [(aa-bb if aa is not None and bb is not None else None) for aa, bb in zip(a, b)]
            tag_used = "derived"
    else:
        vals, tag_used = build_row(facts_gaap, tags, FY_ENDS, negate)
        if tag_used is None:
            missing_is.append(label)
    is_rows[label] = vals
    is_tags[label] = tag_used

df_is = pd.DataFrame(is_rows, index=FY_LABELS).T
df_is.index.name = "Line Item"
miss = f" ({len(missing_is)} missing: {', '.join(missing_is)})" if missing_is else " (0 missing)"
print(f"\u2705 Income Statement: {len(IS_SPEC)} line items mapped{miss}")

In [ ]:
# ── Cell 6: Balance Sheet DataFrame ──────────────────────────────────────────

BS_SPEC = [
    ("Cash & Equivalents",        ["CashAndCashEquivalentsAtCarryingValue",
                                   "CashCashEquivalentsAndShortTermInvestments"],          False),
    ("Short-Term Investments",    ["ShortTermInvestments",
                                   "AvailableForSaleSecuritiesCurrent"],                   False),
    ("Accounts Receivable",       ["AccountsReceivableNetCurrent",
                                   "ReceivablesNetCurrent"],                               False),
    ("Inventory",                 ["InventoryNet", "InventoryGross"],                       False),
    ("Total Current Assets",      ["AssetsCurrent"],                                       False),
    ("PP&E Net",                  ["PropertyPlantAndEquipmentNet",
                                   "PropertyPlantAndEquipmentAndFinanceLeaseRightOfUseAssetAfterAccumulatedDepreciationAndAmortization"], False),
    ("Goodwill",                  ["Goodwill"],                                            False),
    ("Intangible Assets",         ["IntangibleAssetsNetExcludingGoodwill",
                                   "FiniteLivedIntangibleAssetsNet"],                      False),
    ("Total Assets",              ["Assets"],                                              False),
    ("Accounts Payable",          ["AccountsPayableCurrent",
                                   "AccountsPayableAndAccruedLiabilitiesCurrent"],         False),
    ("Short-Term Debt",           ["ShortTermBorrowings", "LongTermDebtCurrent"],          False),
    ("Total Current Liabilities", ["LiabilitiesCurrent"],                                  False),
    ("Long-Term Debt",            ["LongTermDebt", "LongTermDebtNoncurrent"],              False),
    ("Total Liabilities",         ["Liabilities"],                                         False),
    ("Total Equity",              ["StockholdersEquity",
                                   "StockholdersEquityIncludingPortionAttributableToNoncontrollingInterest"], False),
    ("Total Liabilities & Equity",["LiabilitiesAndStockholdersEquity"],                    False),
]

bs_rows, bs_tags, missing_bs = {}, {}, []
for label, tags, negate in BS_SPEC:
    vals, tag_used = build_row(facts_gaap, tags, FY_ENDS, negate)
    bs_rows[label] = vals; bs_tags[label] = tag_used
    if tag_used is None: missing_bs.append(label)

df_bs = pd.DataFrame(bs_rows, index=FY_LABELS).T
df_bs.index.name = "Line Item"
miss = f" ({len(missing_bs)} missing: {', '.join(missing_bs)})" if missing_bs else " (0 missing)"
print(f"\u2705 Balance Sheet: {len(BS_SPEC)} line items mapped{miss}")

In [ ]:
# ── Cell 7: Cash Flow Statement DataFrame ────────────────────────────────────

CFS_SPEC = [
    ("Net Income (CFS)",         ["NetIncomeLoss"],                                        False),
    ("D&A",                      ["DepreciationDepletionAndAmortization",
                                  "DepreciationAndAmortization"],                          False),
    ("Stock-Based Compensation", ["ShareBasedCompensation",
                                  "AllocatedShareBasedCompensationExpense"],               False),
    ("Change in Working Capital",["IncreaseDecreaseInOperatingCapital"],                   False),
    ("Changes in Receivables",   ["IncreaseDecreaseInAccountsReceivable",
                                  "IncreaseDecreaseInReceivables"],                        False),
    ("Changes in Inventory",     ["IncreaseDecreaseInInventories"],                        False),
    ("Changes in Payables",      ["IncreaseDecreaseInAccountsPayable",
                                  "IncreaseDecreaseInAccountsPayableAndAccruedLiabilities"], False),
    ("Operating Cash Flow",      ["NetCashProvidedByUsedInOperatingActivities"],           False),
    ("Capital Expenditures",     ["PaymentsToAcquirePropertyPlantAndEquipment",
                                  "PaymentsForCapitalImprovements"],                       True),
    ("Acquisitions",             ["PaymentsToAcquireBusinessesNetOfCashAcquired",
                                  "PaymentsToAcquireBusinessesAndInterestInAffiliates"],  True),
    ("Investing Cash Flow",      ["NetCashProvidedByUsedInInvestingActivities"],           False),
    ("Debt Issuance",            ["ProceedsFromIssuanceOfLongTermDebt",
                                  "ProceedsFromIssuanceOfDebt"],                           False),
    ("Debt Repayment",           ["RepaymentsOfLongTermDebt", "RepaymentsOfDebt"],        True),
    ("Dividends Paid",           ["PaymentsOfDividends", "PaymentsOfDividendsCommonStock"], True),
    ("Share Repurchases",        ["PaymentsForRepurchaseOfCommonStock"],                   True),
    ("Financing Cash Flow",      ["NetCashProvidedByUsedInFinancingActivities"],           False),
]

cfs_rows, cfs_tags, missing_cfs = {}, {}, []
for label, tags, negate in CFS_SPEC:
    vals, tag_used = build_row(facts_gaap, tags, FY_ENDS, negate)
    cfs_rows[label] = vals; cfs_tags[label] = tag_used
    if tag_used is None: missing_cfs.append(label)

# FCF = Operating CF + CapEx (CapEx is already stored negative)
ocf   = cfs_rows.get("Operating Cash Flow",  [None]*len(FY_ENDS))
capex = cfs_rows.get("Capital Expenditures", [None]*len(FY_ENDS))
cfs_rows["Free Cash Flow"] = [(o+c if o is not None and c is not None else None) for o,c in zip(ocf,capex)]
cfs_tags["Free Cash Flow"] = "derived"

df_cfs = pd.DataFrame(cfs_rows, index=FY_LABELS).T
df_cfs.index.name = "Line Item"
miss = f" ({len(missing_cfs)} missing: {', '.join(missing_cfs)})" if missing_cfs else " (0 missing)"
print(f"\u2705 Cash Flow Statement: {len(CFS_SPEC)+1} line items mapped{miss}")

In [ ]:
# ── Cell 8: Excel Workbook Builder ───────────────────────────────────────────
#
# Column layout per statement sheet
# ─────────────────────────────────
#  A        : left margin  (width 3)
#  B        : row labels   (width 35)
#  C … C+n-1: historical FY data, oldest → newest  (blue actuals)
#  C+n …   : forecast cols FY[n+1]E … (light-blue fill, user-editable)
#
# Stats block BELOW data section
# ───────────────────────────────
#  One row per key line item, cols: label | 5-yr CAGR | 5-yr Avg
#  No columns reserved to the right — forecast extends freely.

# ── Style constants ───────────────────────────────────────────────────────────
NAVY       = "1F3864"
MED_GRAY   = "BFBFBF"
LIGHT_GRAY = "F2F2F2"
SUBTOTAL_C = "E8E8E8"
ALT_ROW    = "FAFAFA"
PROJ_FILL  = "EBF3FB"   # very light blue — forecast cells
WACC_INPUT = "FFF2CC"   # yellow — user input on WACC sheet
WACC_RSLT  = "D9EAD3"   # green  — WACC result

THIN_TOP = Border(top=Side(style="thin"))

FMT_DOLLAR = '$#,##0.0;($#,##0.0);"-"'
FMT_PCT    = '0.0%;(0.0%);"-"'
FMT_EPS    = '$#,##0.00;($#,##0.00);"-"'
FMT_SHARES = '#,##0.0;(#,##0.0);"-"'
FMT_2DP    = '#,##0.00'

N_FY           = len(FY_LABELS)
N_PROJ         = len(PROJ_LABELS)
DATA_START_COL = 3                         # Column C
DATA_END_COL   = DATA_START_COL + N_FY - 1
PROJ_START_COL = DATA_END_COL + 1          # forecast immediately after actuals

def col(n): return get_column_letter(n)

# ── Sheet helpers ─────────────────────────────────────────────────────────────
def set_header(ws, company, stmt_name):
    last_col = col(PROJ_START_COL + N_PROJ - 1)
    ws.row_dimensions[1].height = 8
    ws.merge_cells(f"B2:{last_col}2")
    c = ws["B2"]
    c.value     = f"{company}  |  {stmt_name}"
    c.font      = Font(bold=True, size=14, color="FFFFFF", name="Calibri")
    c.fill      = PatternFill("solid", fgColor=NAVY)
    c.alignment = Alignment(horizontal="left", vertical="center")
    ws.row_dimensions[2].height = 24
    ws["B3"].value = "All figures in $MM unless noted"
    ws["B3"].font  = Font(italic=True, size=9, color="808080", name="Calibri")
    ws.row_dimensions[3].height = 14
    hdr_fill  = PatternFill("solid", fgColor=MED_GRAY)
    hdr_font  = Font(bold=True, size=10, name="Calibri")
    hdr_align = Alignment(horizontal="center")
    for i, fy in enumerate(FY_LABELS):
        c = ws.cell(row=4, column=DATA_START_COL+i, value=fy)
        c.font = hdr_font; c.fill = hdr_fill; c.alignment = hdr_align
    proj_fill = PatternFill("solid", fgColor="D9D9D9")
    for i, pl in enumerate(PROJ_LABELS):
        c = ws.cell(row=4, column=PROJ_START_COL+i, value=pl)
        c.font = Font(bold=True, size=10, name="Calibri", color="595959")
        c.fill = proj_fill; c.alignment = hdr_align

def set_col_widths(ws):
    ws.column_dimensions["A"].width = 3
    ws.column_dimensions["B"].width = 35
    for i in range(N_FY):   ws.column_dimensions[col(DATA_START_COL+i)].width = 13
    for i in range(N_PROJ): ws.column_dimensions[col(PROJ_START_COL+i)].width = 13

SUBTOTALS_IS  = {"Gross Profit", "Operating Income", "Net Income"}
SUBTOTALS_BS  = {"Total Current Assets", "Total Assets", "Total Liabilities",
                 "Total Equity", "Total Liabilities & Equity"}
SUBTOTALS_CFS = {"Operating Cash Flow", "Investing Cash Flow",
                 "Financing Cash Flow", "Free Cash Flow"}
EPS_ROWS    = {"EPS Basic (per share)", "EPS Diluted (per share)"}
SHARES_ROWS = {"Diluted Shares"}

def write_data_rows(ws, df, start_row, subtotals, section_inserts=None):
    """Write rows from df. Returns (next_row, {label: sheet_row})."""
    ins     = {pos: lbl for lbl, pos in (section_inserts or [])}
    row_map = {}
    r       = start_row
    alt     = False

    for idx, label in enumerate(df.index):
        if idx in ins:
            ws.cell(row=r, column=2, value=ins[idx]).font = Font(bold=True, size=10, name="Calibri")
            for cn in range(2, PROJ_START_COL+N_PROJ+1):
                ws.cell(row=r, column=cn).fill = PatternFill("solid", fgColor=LIGHT_GRAY)
            ws.row_dimensions[r].height = 16
            r += 1; alt = False

        is_sub  = label in subtotals
        bg      = SUBTOTAL_C if is_sub else (ALT_ROW if alt else "FFFFFF")
        rfill   = PatternFill("solid", fgColor=bg)
        num_fmt = FMT_EPS if label in EPS_ROWS else FMT_SHARES if label in SHARES_ROWS else FMT_DOLLAR

        lc = ws.cell(row=r, column=2, value=label)
        lc.font = Font(bold=is_sub, size=10, name="Calibri"); lc.fill = rfill

        for i, fy in enumerate(FY_LABELS):
            v  = df.loc[label, fy]
            dc = ws.cell(row=r, column=DATA_START_COL+i,
                         value=(float(v) if pd.notna(v) and v is not None else None))
            dc.number_format = num_fmt
            dc.font      = Font(color="0000FF", name="Calibri", size=10, bold=is_sub)
            dc.fill      = rfill
            dc.alignment = Alignment(horizontal="right")
            if is_sub: dc.border = THIN_TOP

        # Forecast cells — light-blue fill, unlocked for user input
        pfill = PatternFill("solid", fgColor=PROJ_FILL)
        for i in range(N_PROJ):
            pc = ws.cell(row=r, column=PROJ_START_COL+i)
            pc.number_format = num_fmt
            pc.font = Font(color="000000", name="Calibri", size=10)
            pc.fill = pfill
            pc.alignment = Alignment(horizontal="right")

        row_map[label] = r
        ws.row_dimensions[r].height = 15
        r += 1; alt = not alt

    return r, row_map

def write_section_hdr(ws, r, label, fg=LIGHT_GRAY):
    ws.cell(row=r, column=2, value=label).font = Font(bold=True, size=10, name="Calibri")
    for cn in range(2, PROJ_START_COL+N_PROJ+1):
        ws.cell(row=r, column=cn).fill = PatternFill("solid", fgColor=fg)
    ws.row_dimensions[r].height = 16

def write_formula_row(ws, r, label, fn, num_fmt=FMT_PCT):
    ws.cell(row=r, column=2, value=label).font = Font(size=10, name="Calibri")
    for i in range(N_FY):
        dc = ws.cell(row=r, column=DATA_START_COL+i, value=fn(r, DATA_START_COL+i))
        dc.number_format = num_fmt
        dc.font      = Font(color="000000", name="Calibri", size=10)
        dc.alignment = Alignment(horizontal="right")
    ws.row_dimensions[r].height = 15

def write_stats_block(ws, start_r, row_map, stat_labels, fmt_map=None):
    """
    Summary statistics below data.
    Col B = line item  |  Col C = 5-yr CAGR  |  Col D = 5-yr Avg
    Stats sit beneath the data; no columns consumed to the right.
    """
    write_section_hdr(ws, start_r, "SUMMARY STATISTICS", fg="DEE6EF")
    hdr_r = start_r + 1
    for cn, txt in [(2, "Line Item"), (3, "5-Yr CAGR"), (4, f"5-Yr Avg  ({FY_LABELS[0]}\u2013{FY_LABELS[-1]})")]:
        c = ws.cell(row=hdr_r, column=cn, value=txt)
        c.font = Font(bold=True, size=9, name="Calibri", color="FFFFFF")
        c.fill = PatternFill("solid", fgColor="2E5F8A")
        c.alignment = Alignment(horizontal="center")
    ws.row_dimensions[hdr_r].height = 14

    c_s = col(DATA_START_COL); c_e = col(DATA_END_COL)
    n_p = N_FY - 1 if N_FY > 1 else 1
    r   = hdr_r + 1
    alt = False

    for label in stat_labels:
        dr = row_map.get(label)
        if dr is None: continue
        fill    = PatternFill("solid", fgColor=ALT_ROW if alt else "FFFFFF")
        num_fmt = (fmt_map or {}).get(label, FMT_DOLLAR)

        lc = ws.cell(row=r, column=2, value=label)
        lc.font = Font(size=9, name="Calibri"); lc.fill = fill

        cc = ws.cell(row=r, column=3,
                     value=f"=IFERROR(({c_e}{dr}/{c_s}{dr})^(1/{n_p})-1,\"-\")")
        cc.number_format = FMT_PCT
        cc.font      = Font(bold=True, size=9, name="Calibri")
        cc.fill      = fill
        cc.alignment = Alignment(horizontal="center")

        ac = ws.cell(row=r, column=4,
                     value=f"=IFERROR(AVERAGE({c_s}{dr}:{c_e}{dr}),\"-\")")
        ac.number_format = num_fmt
        ac.font      = Font(bold=True, size=9, name="Calibri")
        ac.fill      = fill
        ac.alignment = Alignment(horizontal="right")

        ws.row_dimensions[r].height = 14
        r += 1; alt = not alt

    ws.column_dimensions[col(3)].width = max(ws.column_dimensions[col(3)].width or 0, 12)
    ws.column_dimensions[col(4)].width = max(ws.column_dimensions[col(4)].width or 0, 18)
    return r

# ─────────────────────────────────────────────────────────────────────────────
wb = Workbook()
wb.remove(wb.active)

# ── COVER SHEET ───────────────────────────────────────────────────────────────
cover = wb.create_sheet("Cover")
cover.column_dimensions["A"].width = 4
cover.column_dimensions["B"].width = 30
cover.column_dimensions["C"].width = 44

cover.merge_cells("B2:D2")
t = cover["B2"]
t.value     = f"{company_name}  |  SEC 10-K Financial Model"
t.font      = Font(bold=True, size=16, color="FFFFFF", name="Calibri")
t.fill      = PatternFill("solid", fgColor=NAVY)
t.alignment = Alignment(horizontal="left", vertical="center")
cover.row_dimensions[2].height = 30

for i, (k, v) in enumerate([
    ("Company",       company_name),
    ("Ticker",        TICKER.upper()),
    ("CIK",           CIK),
    ("Date Pulled",   TODAY),
    ("Fiscal Years",  ", ".join(FY_LABELS)),
    ("Forecast Cols", ", ".join(PROJ_LABELS)),
    ("Units",         "All values in $MM unless per-share"),
    ("Data Source",   "https://data.sec.gov  (XBRL company facts)"),
], start=4):
    cover.cell(row=i, column=2, value=k).font = Font(bold=True, name="Calibri", size=10)
    cover.cell(row=i, column=3, value=v).font = Font(name="Calibri", size=10)

cover.cell(row=13, column=2, value="Contents").font = Font(bold=True, size=11, name="Calibri")
for i, sname in enumerate(["Income Statement", "Balance Sheet",
                            "Cash Flow Statement", "WACC", "Raw SEC Facts"], start=14):
    c = cover.cell(row=i, column=2, value=sname)
    c.hyperlink = f"#{sname}!A1"
    c.font = Font(color="0563C1", underline="single", name="Calibri", size=10)

# ── INCOME STATEMENT SHEET ────────────────────────────────────────────────────
ws_is = wb.create_sheet("Income Statement")
set_col_widths(ws_is)
set_header(ws_is, company_name, "Income Statement")

next_r, is_map = write_data_rows(
    ws_is, df_is, start_row=5, subtotals=SUBTOTALS_IS,
    section_inserts=[("REVENUE", 0), ("OPERATING EXPENSES", 2),
                     ("PROFITABILITY", 5), ("PER SHARE & OTHER", 10)])

# Margins & Ratios
next_r += 1; write_section_hdr(ws_is, next_r, "MARGINS & RATIOS"); next_r += 1
rv = is_map.get("Revenue");          gp  = is_map.get("Gross Profit")
oi = is_map.get("Operating Income"); ni  = is_map.get("Net Income")
rd = is_map.get("R&D Expense");      sg  = is_map.get("SG&A Expense")
tx = is_map.get("Income Tax Expense"); pt = is_map.get("Pretax Income")

def mfn(nr, dr): return lambda r, c: f"=IFERROR({col(c)}{nr}/{col(c)}{dr},\"-\")"

for lbl, fn in [("Gross Margin %", mfn(gp, rv)), ("Operating Margin %", mfn(oi, rv)),
                ("Net Margin %",    mfn(ni, rv)), ("R&D % of Revenue",   mfn(rd, rv)),
                ("SG&A % of Revenue", mfn(sg, rv)), ("Effective Tax Rate %", mfn(tx, pt))]:
    write_formula_row(ws_is, next_r, lbl, fn, FMT_PCT); next_r += 1

# YoY Revenue Growth
ws_is.cell(row=next_r, column=2, value="YoY Revenue Growth %").font = Font(size=10, name="Calibri")
for i in range(N_FY):
    if i == 0: continue
    dc = ws_is.cell(row=next_r, column=DATA_START_COL+i,
                    value=f"=IFERROR({col(DATA_START_COL+i)}{rv}/{col(DATA_START_COL+i-1)}{rv}-1,\"-\")")
    dc.number_format = FMT_PCT
    dc.font = Font(color="000000", name="Calibri", size=10)
    dc.alignment = Alignment(horizontal="right")
ws_is.row_dimensions[next_r].height = 15; next_r += 1

# Summary stats below
next_r += 1
next_r = write_stats_block(ws_is, next_r, is_map,
    ["Revenue", "Gross Profit", "Operating Income", "Net Income", "EPS Diluted (per share)"],
    {"EPS Diluted (per share)": FMT_EPS})

# ── BALANCE SHEET SHEET ───────────────────────────────────────────────────────
ws_bs = wb.create_sheet("Balance Sheet")
set_col_widths(ws_bs)
set_header(ws_bs, company_name, "Balance Sheet")

next_r_bs, bs_map = write_data_rows(
    ws_bs, df_bs, start_row=5, subtotals=SUBTOTALS_BS,
    section_inserts=[("CURRENT ASSETS", 0), ("NON-CURRENT ASSETS", 5),
                     ("CURRENT LIABILITIES", 9), ("NON-CURRENT LIABILITIES", 12), ("EQUITY", 14)])

# Key Ratios
next_r_bs += 1; write_section_hdr(ws_bs, next_r_bs, "KEY RATIOS"); next_r_bs += 1
ca = bs_map.get("Total Current Assets");  cl  = bs_map.get("Total Current Liabilities")
te = bs_map.get("Total Equity");          std = bs_map.get("Short-Term Debt")
ltd = bs_map.get("Long-Term Debt");       csh = bs_map.get("Cash & Equivalents")

for lbl, fn, fmt in [
    ("Current Ratio",  lambda r,c: f"=IFERROR({col(c)}{ca}/{col(c)}{cl},\"-\")",  FMT_2DP),
    ("Debt-to-Equity", lambda r,c: f"=IFERROR(({col(c)}{std}+{col(c)}{ltd})/{col(c)}{te},\"-\")", FMT_2DP),
    ("Net Debt ($MM)", lambda r,c: f"=IFERROR(({col(c)}{std}+{col(c)}{ltd})-{col(c)}{csh},\"-\")", FMT_DOLLAR),
]:
    write_formula_row(ws_bs, next_r_bs, lbl, fn, fmt); next_r_bs += 1

# Summary stats below
next_r_bs += 1
next_r_bs = write_stats_block(ws_bs, next_r_bs, bs_map,
    ["Total Current Assets", "Total Assets", "Total Current Liabilities",
     "Total Liabilities", "Total Equity"])

# ── CASH FLOW STATEMENT SHEET ─────────────────────────────────────────────────
ws_cfs = wb.create_sheet("Cash Flow Statement")
set_col_widths(ws_cfs)
set_header(ws_cfs, company_name, "Cash Flow Statement")

next_r_cfs, cfs_map = write_data_rows(
    ws_cfs, df_cfs, start_row=5, subtotals=SUBTOTALS_CFS,
    section_inserts=[("OPERATING ACTIVITIES", 0), ("INVESTING ACTIVITIES", 8),
                     ("FINANCING ACTIVITIES", 11)])

# Key Metrics
next_r_cfs += 1; write_section_hdr(ws_cfs, next_r_cfs, "KEY METRICS"); next_r_cfs += 1
ocf_r = cfs_map.get("Operating Cash Flow"); fcf_r = cfs_map.get("Free Cash Flow")
cpx_r = cfs_map.get("Capital Expenditures"); ni_c  = cfs_map.get("Net Income (CFS)")
IS_SH = "'Income Statement'"

for lbl, fn, fmt in [
    ("FCF Margin %",       lambda r,c: f"=IFERROR({col(c)}{fcf_r}/{IS_SH}!{col(c)}{rv},\"-\")", FMT_PCT),
    ("CapEx % of Revenue", lambda r,c: f"=IFERROR(ABS({col(c)}{cpx_r})/{IS_SH}!{col(c)}{rv},\"-\")", FMT_PCT),
    ("FCF Conversion %",   lambda r,c: f"=IFERROR({col(c)}{fcf_r}/{col(c)}{ni_c},\"-\")", FMT_PCT),
]:
    write_formula_row(ws_cfs, next_r_cfs, lbl, fn, fmt); next_r_cfs += 1

# Summary stats below
next_r_cfs += 1
next_r_cfs = write_stats_block(ws_cfs, next_r_cfs, cfs_map,
    ["Operating Cash Flow", "Capital Expenditures",
     "Investing Cash Flow", "Financing Cash Flow", "Free Cash Flow"])

# ── WACC SHEET ────────────────────────────────────────────────────────────────
#
# Structure mirrors the DCFWacc template:
#   Cost of Debt section
#   Cost of Equity section (CAPM)
#   Capital Structure section
#   WACC result (green highlight)
#   Sensitivity table: WACC vs Beta & ERP
#
ws_w = wb.create_sheet("WACC")
ws_w.column_dimensions["A"].width = 4
ws_w.column_dimensions["B"].width = 38
ws_w.column_dimensions["C"].width = 16
ws_w.column_dimensions["D"].width = 16
ws_w.column_dimensions["E"].width = 13
ws_w.column_dimensions["F"].width = 13
ws_w.column_dimensions["G"].width = 13

def w_title(r, text):
    ws_w.merge_cells(f"B{r}:G{r}")
    c = ws_w[f"B{r}"]
    c.value = text
    c.font  = Font(bold=True, size=14, color="FFFFFF", name="Calibri")
    c.fill  = PatternFill("solid", fgColor=NAVY)
    c.alignment = Alignment(horizontal="left", vertical="center")
    ws_w.row_dimensions[r].height = 26

def w_sec(r, text):
    ws_w.merge_cells(f"B{r}:G{r}")
    c = ws_w[f"B{r}"]
    c.value = text
    c.font  = Font(bold=True, size=10, color="FFFFFF", name="Calibri")
    c.fill  = PatternFill("solid", fgColor="2E5F8A")
    ws_w.row_dimensions[r].height = 16

def w_row(r, label, value=None, formula=None, fmt=FMT_PCT, is_input=False, is_result=False):
    lc = ws_w.cell(row=r, column=2, value=label)
    lc.font = Font(bold=is_result, size=10, name="Calibri")
    if is_result: lc.fill = PatternFill("solid", fgColor=WACC_RSLT)
    vc = ws_w.cell(row=r, column=3, value=formula if formula else value)
    vc.number_format = fmt
    vc.alignment = Alignment(horizontal="center")
    if is_input:
        vc.fill = PatternFill("solid", fgColor=WACC_INPUT)
        vc.font = Font(color="0000FF", size=10, name="Calibri")
    elif is_result:
        vc.fill = PatternFill("solid", fgColor=WACC_RSLT)
        vc.font = Font(bold=True, color="375623", size=11, name="Calibri")
    else:
        vc.font = Font(color="000000", size=10, name="Calibri")
    ws_w.row_dimensions[r].height = 15

w_title(2, f"{company_name}  |  WACC Buildup")
ws_w["B3"].value = "Yellow cells = user inputs.  Green cell = WACC result (referenced by DCF model)."
ws_w["B3"].font  = Font(italic=True, size=9, color="808080", name="Calibri")

# Cost of Debt (rows 5-8)
w_sec(5, "COST OF DEBT")
w_row(6,  "Pre-tax cost of debt  (Kd)",             value=0.045, fmt=FMT_PCT, is_input=True)
w_row(7,  "Marginal tax rate  (T)",                  value=0.21,  fmt=FMT_PCT, is_input=True)
w_row(8,  "After-tax cost of debt  Kd × (1 − T)",   formula="=C6*(1-C7)", fmt=FMT_PCT)

# Cost of Equity — CAPM (rows 10-14)
w_sec(10, "COST OF EQUITY  —  CAPM")
w_row(11, "Risk-free rate  (Rf)  — 10-yr UST",      value=0.043,  fmt=FMT_PCT, is_input=True)
w_row(12, "Equity beta  (β)",                        value=1.20,   fmt=FMT_2DP, is_input=True)
w_row(13, "Equity risk premium  (ERP)",               value=0.055,  fmt=FMT_PCT, is_input=True)
w_row(14, "Cost of equity  Ke = Rf + β × ERP",       formula="=C11+C12*C13", fmt=FMT_PCT)

# Capital Structure (rows 16-20)
w_sec(16, "CAPITAL STRUCTURE")
for cn, txt, w in [(2,"Component",38),(3,"Market Value ($MM)",16),(4,"Target Override",16),(5,"Weight",13)]:
    c = ws_w.cell(row=17, column=cn, value=txt)
    c.font      = Font(bold=True, size=9, name="Calibri")
    c.fill      = PatternFill("solid", fgColor=MED_GRAY)
    c.alignment = Alignment(horizontal="center")
ws_w.row_dimensions[17].height = 14

for r_, lbl, we_formula in [
    (18, "Equity (Market Capitalization)",   "=IF(D18<>\"\",D18,C18/(C18+C19))"),
    (19, "Net Debt  (Total Debt − Cash)",    "=1-E18"),
]:
    ws_w.cell(row=r_, column=2, value=lbl).font = Font(size=10, name="Calibri")
    inp = ws_w.cell(row=r_, column=3)
    inp.fill = PatternFill("solid", fgColor=WACC_INPUT)
    inp.font = Font(color="0000FF", size=10, name="Calibri")
    inp.number_format = FMT_DOLLAR
    inp.alignment = Alignment(horizontal="center")
    we = ws_w.cell(row=r_, column=5, value=we_formula)
    we.number_format = FMT_PCT
    we.font = Font(color="000000", size=10, name="Calibri")
    we.alignment = Alignment(horizontal="center")
    ws_w.row_dimensions[r_].height = 15

# WACC result (row 22)
w_sec(21, "WACC")
ws_w.cell(row=22, column=2, value="WACC  =  Ke × We  +  Kd(1−T) × Wd").font = Font(bold=True, size=11, name="Calibri")
ws_w.cell(row=22, column=2).fill = PatternFill("solid", fgColor=WACC_RSLT)
wr = ws_w.cell(row=22, column=3, value="=C14*E18+C8*E19")
wr.number_format = FMT_PCT
wr.font      = Font(bold=True, size=13, color="375623", name="Calibri")
wr.fill      = PatternFill("solid", fgColor=WACC_RSLT)
wr.alignment = Alignment(horizontal="center", vertical="center")
ws_w.row_dimensions[22].height = 22

# Sensitivity table: WACC vs Beta (columns) and ERP (rows)  —  rows 25-32
w_sec(25, "SENSITIVITY  —  WACC  vs  Beta  &  Equity Risk Premium")
ws_w.cell(row=26, column=2, value="WACC").font = Font(bold=True, size=9, name="Calibri")
ws_w.cell(row=26, column=3, value="Beta →").font  = Font(bold=True, size=9, name="Calibri", color="595959")
ws_w.cell(row=27, column=2, value="ERP ↓").font   = Font(bold=True, size=9, name="Calibri", color="595959")

BETA_OFFSETS = [-0.30, -0.15, 0.00, 0.15, 0.30]
ERP_OFFSETS  = [-0.010, -0.005, 0.000, 0.005, 0.010]

for j, bo in enumerate(BETA_OFFSETS):
    beta_f = "=C12" if bo == 0 else (f"={bo:+.2f}+C12")
    c = ws_w.cell(row=26, column=3+j, value=beta_f)
    c.number_format = FMT_2DP
    c.font = Font(bold=True, size=9, name="Calibri")
    c.fill = PatternFill("solid", fgColor=MED_GRAY)
    c.alignment = Alignment(horizontal="center")

for i, eo in enumerate(ERP_OFFSETS):
    erp_f = "=C13" if eo == 0 else (f"={eo:+.3f}+C13")
    er = ws_w.cell(row=27+i, column=2, value=erp_f)
    er.number_format = FMT_PCT
    er.font = Font(bold=True, size=9, name="Calibri")
    er.fill = PatternFill("solid", fgColor=MED_GRAY)
    er.alignment = Alignment(horizontal="center")
    for j, bo in enumerate(BETA_OFFSETS):
        beta_expr = "C12" if bo == 0 else f"({bo:+.2f}+C12)"
        erp_expr  = "C13" if eo == 0 else f"({eo:+.3f}+C13)"
        f = f"=(C11+{beta_expr}*{erp_expr})*E18+C8*E19"
        vc = ws_w.cell(row=27+i, column=3+j, value=f)
        vc.number_format = FMT_PCT
        vc.font = Font(size=9, name="Calibri")
        vc.alignment = Alignment(horizontal="center")
        if i == 2 and j == 2:   # base case
            vc.fill = PatternFill("solid", fgColor=WACC_RSLT)
            vc.font = Font(bold=True, size=9, color="375623", name="Calibri")
    ws_w.row_dimensions[27+i].height = 14

# Notes
ws_w.cell(row=34, column=2, value="Notes").font = Font(bold=True, size=9, name="Calibri")
for i, note in enumerate([
    "\u2022 Yellow cells are user inputs. Update Rf, \u03b2, ERP, market cap, and net debt for this company.",
    "\u2022 Risk-free rate: current 10-year US Treasury yield.",
    "\u2022 ERP: Damodaran estimates ~5.5% for US equities (implied ERP, Jan 2025).",
    "\u2022 Beta: observed 5-year monthly beta vs. S&P 500 (Bloomberg, Yahoo Finance).",
    "\u2022 Market cap and net debt: pull from the Balance Sheet tab or live market data.",
    "\u2022 Cell C22 (WACC result) can be referenced directly in a DCF model.",
], start=35):
    ws_w.cell(row=i, column=2, value=note).font = Font(size=9, name="Calibri", color="595959")
    ws_w.row_dimensions[i].height = 13

# ── RAW SEC FACTS SHEET ───────────────────────────────────────────────────────
ws_raw = wb.create_sheet("Raw SEC Facts")
ws_raw.freeze_panes = "A2"
ws_raw.append(["Statement", "Line Item", "Tag Used"] + FY_LABELS)

raw_rows = []
for stmt, df_map, tag_map in [
    ("Income Statement",    df_is,  is_tags),
    ("Balance Sheet",       df_bs,  bs_tags),
    ("Cash Flow Statement", df_cfs, cfs_tags),
]:
    for label in df_map.index:
        row = [stmt, label, tag_map.get(label, "")]
        for fy in FY_LABELS:
            v = df_map.loc[label, fy]
            row.append(float(v) if pd.notna(v) and v is not None else None)
        ws_raw.append(row); raw_rows.append(row)

n_raw = len(raw_rows) + 1
tbl   = Table(displayName="RawSECFacts", ref=f"A1:{col(3+N_FY)}{n_raw}")
tbl.tableStyleInfo = TableStyleInfo(name="TableStyleMedium9", showRowStripes=True)
ws_raw.add_table(tbl)
ws_raw.column_dimensions["A"].width = 20
ws_raw.column_dimensions["B"].width = 30
ws_raw.column_dimensions["C"].width = 55
for i in range(N_FY): ws_raw.column_dimensions[col(4+i)].width = 13

print("\u2705 Excel workbook built successfully")

In [ ]:
# ── Cell 9: Save File & Print Summary ────────────────────────────────────────

wb.save(OUTPUT_FILE)
print(f"\u2705 Excel workbook saved: {OUTPUT_FILE}")
print()
print("Summary")
print("-------")
print(f"  Company          : {company_name}")
print(f"  Ticker           : {TICKER.upper()}")
print(f"  CIK              : {CIK}")
print(f"  Fiscal Years     : {', '.join(FY_LABELS)}")
print(f"  Forecast Columns : {', '.join(PROJ_LABELS)}")
print(f"  IS rows          : {len(df_is)}")
print(f"  BS rows          : {len(df_bs)}")
print(f"  CFS rows         : {len(df_cfs)}")
print(f"  Output file      : {OUTPUT_FILE}")